# 图像卷积
---
## 环境配置

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), ".."))
os.environ["TILE_FWK_DEVICE_ID"] = "0"
import pypto
import torch
import torch_npu
import numpy as np

device_id = int(os.environ["TILE_FWK_DEVICE_ID"])
torch.npu.set_device(device_id)
device = f"npu:{device_id}"
mode = pypto.RunMode.NPU

---

## 练习 6.2.1  

构建一个具有对角线边缘的图像`X`。
1. 如果将本节中举例的卷积核`K`应用于`X`，会发生什么情况？
2. 如果转置`X`会发生什么？
3. 如果转置`K`会发生什么？

### 解答

**第1问：**

&emsp;&emsp;在对角线处有分别为 $1$ 和 $-1$ 的数据，其他区域都为 $0$。

以下使用 `torch` 编程进行验证：

In [3]:
import torch
from torch import nn
from src.D2LFunction import *

def corr2d(X, K):
    """计算二维互相关运算"""
    h, w = K.shape
    Y = torch.zeros((X.shape[0] - h + 1, X.shape[1] - w + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = (X[i:i + h, j:j + w] * K).sum()
    return Y

X = torch.eye(8)
K = torch.tensor([[1.0, -1.0]])
Y = corr2d(X, K)
print(Y)

tensor([[ 1.,  0.,  0.,  0.,  0.,  0.,  0.],
        [-1.,  1.,  0.,  0.,  0.,  0.,  0.],
        [ 0., -1.,  1.,  0.,  0.,  0.,  0.],
        [ 0.,  0., -1.,  1.,  0.,  0.,  0.],
        [ 0.,  0.,  0., -1.,  1.,  0.,  0.],
        [ 0.,  0.,  0.,  0., -1.,  1.,  0.],
        [ 0.,  0.,  0.,  0.,  0., -1.,  1.],
        [ 0.,  0.,  0.,  0.,  0.,  0., -1.]])


使用 `PyPTO` 编程进行验证:

In [4]:
from src.PyPTOConvPrimitive import PyPTOConv2d

X = torch.eye(8, dtype=torch.float32, device=device)
K = torch.tensor([[1.0, -1.0]], dtype=torch.float32, device=device)
conv2d = PyPTOConv2d(in_channels=1, out_channels=1, kernel_size=(1, 2),
                     bias=False, device=device)
conv2d.weight.data = K.reshape(1, 1, 1, 2)
X_4d = X.reshape(1, 1, 8, 8)
Y = conv2d(X_4d)
out = Y.reshape(8, 7)
print(out)

tensor([[ 1.,  0.,  0.,  0.,  0.,  0.,  0.],
        [-1.,  1.,  0.,  0.,  0.,  0.,  0.],
        [ 0., -1.,  1.,  0.,  0.,  0.,  0.],
        [ 0.,  0., -1.,  1.,  0.,  0.,  0.],
        [ 0.,  0.,  0., -1.,  1.,  0.,  0.],
        [ 0.,  0.,  0.,  0., -1.,  1.,  0.],
        [ 0.,  0.,  0.,  0.,  0., -1.,  1.],
        [ 0.,  0.,  0.,  0.,  0.,  0., -1.]], device='npu:0',
       grad_fn=<ViewBackward0>)


**第2问：**

&emsp;&emsp;`X`转置后，结果不变。

以下使用 `torch` 编程进行验证：

In [5]:
import torch
from torch import nn
from src.D2LFunction import *

def corr2d(X, K):
    """计算二维互相关运算"""
    h, w = K.shape
    Y = torch.zeros((X.shape[0] - h + 1, X.shape[1] - w + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = (X[i:i + h, j:j + w] * K).sum()
    return Y

X = torch.eye(8)
K = torch.tensor([[1.0, -1.0]])
Y = corr2d(X.T, K)
print(Y)

tensor([[ 1.,  0.,  0.,  0.,  0.,  0.,  0.],
        [-1.,  1.,  0.,  0.,  0.,  0.,  0.],
        [ 0., -1.,  1.,  0.,  0.,  0.,  0.],
        [ 0.,  0., -1.,  1.,  0.,  0.,  0.],
        [ 0.,  0.,  0., -1.,  1.,  0.,  0.],
        [ 0.,  0.,  0.,  0., -1.,  1.,  0.],
        [ 0.,  0.,  0.,  0.,  0., -1.,  1.],
        [ 0.,  0.,  0.,  0.,  0.,  0., -1.]])


使用 `PyPTO` 编程进行验证:

In [6]:
from src.PyPTOConvPrimitive import PyPTOConv2d

X = torch.eye(8, dtype=torch.float32, device=device)
K = torch.tensor([[1.0, -1.0]], dtype=torch.float32, device=device)
conv2d = PyPTOConv2d(in_channels=1, out_channels=1, kernel_size=(1, 2),
                     bias=False, device=device)
conv2d.weight.data = K.reshape(1, 1, 1, 2)
X_4d = X.T.reshape(1, 1, 8, 8)
Y = conv2d(X_4d)
out = Y.reshape(8, 7)
print(out)

tensor([[ 1.,  0.,  0.,  0.,  0.,  0.,  0.],
        [-1.,  1.,  0.,  0.,  0.,  0.,  0.],
        [ 0., -1.,  1.,  0.,  0.,  0.,  0.],
        [ 0.,  0., -1.,  1.,  0.,  0.,  0.],
        [ 0.,  0.,  0., -1.,  1.,  0.,  0.],
        [ 0.,  0.,  0.,  0., -1.,  1.,  0.],
        [ 0.,  0.,  0.,  0.,  0., -1.,  1.],
        [ 0.,  0.,  0.,  0.,  0.,  0., -1.]], device='npu:0',
       grad_fn=<ViewBackward0>)


**第3问：**

&emsp;&emsp;`K`转置后，结果也会发生转置。

以下使用 `torch` 编程进行验证：

In [7]:
import torch
from torch import nn
from src.D2LFunction import *

def corr2d(X, K):
    """计算二维互相关运算"""
    h, w = K.shape
    Y = torch.zeros((X.shape[0] - h + 1, X.shape[1] - w + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = (X[i:i + h, j:j + w] * K).sum()
    return Y

X = torch.eye(8)
K = torch.tensor([[1.0, -1.0]])
Y = corr2d(X, K.T)
print(Y)

tensor([[ 1., -1.,  0.,  0.,  0.,  0.,  0.,  0.],
        [ 0.,  1., -1.,  0.,  0.,  0.,  0.,  0.],
        [ 0.,  0.,  1., -1.,  0.,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  1., -1.,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  0.,  1., -1.,  0.,  0.],
        [ 0.,  0.,  0.,  0.,  0.,  1., -1.,  0.],
        [ 0.,  0.,  0.,  0.,  0.,  0.,  1., -1.]])


使用 `PyPTO` 编程进行验证:

In [8]:
from src.PyPTOConvPrimitive import PyPTOConv2d

X = torch.eye(8, dtype=torch.float32, device=device)
K = torch.tensor([[1.0, -1.0]], dtype=torch.float32, device=device)
conv2d = PyPTOConv2d(in_channels=1, out_channels=1, kernel_size=(2, 1),
                     bias=False, device=device)
conv2d.weight.data = K.T.reshape(1, 1, 2, 1)
X_4d = X.reshape(1, 1, 8, 8)
Y = conv2d(X_4d)
out = Y.reshape(7, 8)
print(out)

tensor([[ 1., -1.,  0.,  0.,  0.,  0.,  0.,  0.],
        [ 0.,  1., -1.,  0.,  0.,  0.,  0.,  0.],
        [ 0.,  0.,  1., -1.,  0.,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  1., -1.,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  0.,  1., -1.,  0.,  0.],
        [ 0.,  0.,  0.,  0.,  0.,  1., -1.,  0.],
        [ 0.,  0.,  0.,  0.,  0.,  0.,  1., -1.]], device='npu:0',
       grad_fn=<ViewBackward0>)


---

## 练习 6.2.2 

在我们创建的`Conv2D`自动求导时，有什么错误消息？

### 解答

以下使用 `torch` 编程进行验证：

In [9]:
import torch
from torch import nn
from src.D2LFunction import *

def corr2d(X, K):
    """计算二维互相关运算"""
    h, w = K.shape
    Y = torch.zeros((X.shape[0] - h + 1, X.shape[1] - w + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = (X[i:i + h, j:j + w] * K).sum()
    return Y

class Conv2D(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.weight = nn.Parameter(torch.rand(kernel_size))
        self.bias = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        return corr2d(x, self.weight) + self.bias

X = torch.ones((6, 8))
X[:, 2:6] = 0
K = torch.tensor([[1.0, -1.0]])
Y = corr2d(X, K)

使用 `nn.Conv2d`（框架自带，4D 输入，正常训练）：

In [10]:
conv2d = nn.Conv2d(1, 1, kernel_size=(1, 2), bias=False)
X_4d = X.reshape((1, 1, 6, 8))
Y_4d = Y.reshape((1, 1, 6, 7))
lr = 3e-2

for i in range(10):
    Y_hat = conv2d(X_4d)
    l = (Y_hat - Y_4d) ** 2
    conv2d.zero_grad()
    l.sum().backward()
    conv2d.weight.data[:] -= lr * conv2d.weight.grad
    if (i + 1) % 2 == 0:
        print(f'epoch {i+1}, loss {l.sum():.3f}')

epoch 2, loss 14.939
epoch 4, loss 4.433
epoch 6, loss 1.533
epoch 8, loss 0.580
epoch 10, loss 0.230


使用自定义 `Conv2D`（4D 输入，报维度不匹配错误）：

In [11]:
conv2d = Conv2D(kernel_size=(1, 2))
try:
    conv2d(X_4d)
except Exception as e:
    print(e)

修复：使用 2D 输入即可正常训练：

In [12]:
conv2d = Conv2D(kernel_size=(1, 2))
lr = 3e-2

for i in range(10):
    Y_hat = conv2d(X)
    l = (Y_hat - Y) ** 2
    conv2d.zero_grad()
    l.sum().backward()
    conv2d.weight.data[:] -= lr * conv2d.weight.grad
    if (i + 1) % 2 == 0:
        print(f'epoch {i+1}, loss {l.sum():.3f}')

epoch 2, loss 29.813
epoch 4, loss 11.091
epoch 6, loss 4.355
epoch 8, loss 1.752
epoch 10, loss 0.712


使用 `PyPTO` 编程进行验证:

以下使用 notebook 中实现的 `PyPTOConv2d`（基于 `corr2d_kernel` + `torch.autograd.Function`）进行二维卷积验证。该实现不依赖 `pypto.conv` 算子，通过 `pypto.loop` + `pypto.view` + `pypto.assemble` 完成互相关运算。

In [13]:
from src.PyPTOConvPrimitive import corr2d, PyPTOConv2d

X = torch.ones((6, 8), dtype=torch.float32, device=device)
X[:, 2:6] = 0
K = torch.tensor([[1.0, -1.0]], dtype=torch.float32, device=device)
Y = corr2d(X, K)

# 原始 corr2d 只接受 2D 张量, 传入 4D 会触发错误
X_4d = X.reshape((1, 1, 6, 8))
try:
    corr2d(X_4d, K)
except Exception as e:
    print(f'corr2d 4D error: {e}')

# PyPTOConv2d 原生支持 4D, 不会报错
conv2d = PyPTOConv2d(in_channels=1, out_channels=1, kernel_size=(1, 2),
                     bias=False, device=device)
out = conv2d(X_4d)
print(f'PyPTOConv2d 4D output: {out.shape}')

# 用 2D 数据做训练 (与 PyTorch 参考答案一致)
conv2d_2d = PyPTOConv2d(in_channels=1, out_channels=1, kernel_size=(1, 2),
                        bias=False, device=device)
lr = 3e-2
Y_hat = conv2d_2d(X_4d)
Y_4d = Y.reshape(1, 1, 6, 7)
for i in range(10):
    Y_hat = conv2d_2d(X_4d)
    l = (Y_hat - Y_4d) ** 2
    conv2d_2d.zero_grad()
    l.sum().backward()
    conv2d_2d.weight.data[:] -= lr * conv2d_2d.weight.grad
    if (i + 1) % 2 == 0:
        print(f'epoch {i+1}, loss {l.sum():.3f}')

corr2d 4D error: too many values to unpack (expected 4)
PyPTOConv2d 4D output: torch.Size([1, 1, 6, 7])


epoch 2, loss 12.320
epoch 4, loss 3.038
epoch 6, loss 0.908
epoch 8, loss 0.315
epoch 10, loss 0.120


---

## 练习 6.2.3

如何通过改变输入张量和卷积核张量，将互相关运算表示为矩阵乘法？

### 解答

&emsp;&emsp;题目的意思应该是如何通过矩阵乘法得到互相关（卷积）运算。

以下使用 `torch` 编程进行验证：

In [14]:
import torch
from torch import nn
from src.D2LFunction import *

def conv2d_by_mul(X, K):
    h, w = K.shape
    outh = X.shape[0] - h + 1
    outw = X.shape[1] - w + 1
    K = K.reshape(-1, 1)
    Y = []
    for i in range(outh):
        for j in range(outw):
            Y.append(X[i:i + h, j:j + w].reshape(-1))
    Y = torch.stack(Y, 0)
    res = (torch.matmul(Y, K)).reshape(outh, outw)
    return res

X = torch.ones((2, 3))
X[:, 1] = 0
print(X)
K = torch.ones((2, 3))
output = conv2d_by_mul(X, K)
print(output)

tensor([[1., 0., 1.],
        [1., 0., 1.]])
tensor([[4.]])


使用 `PyPTO` 编程进行验证:

In [15]:
from src.PyPTOConvPrimitive import corr2d

# PyPTO: 用 pypto.matmul 实现矩阵乘法版互相关
# 思路与 PyTorch 的 conv2d_by_mul 一致:
# 1. 将输入图像按卷积核大小切分为 patch，展平为行向量，堆叠为矩阵 (im2col)
# 2. 将卷积核展平为列向量
# 3. matmul(patch_matrix, kernel_vector) = 互相关结果

@pypto.frontend.jit(runtime_options={"run_mode": mode})
def practice_6_2_3_kernel(
    col_mat: pypto.Tensor([], pypto.DT_FP32),
    k_vec:   pypto.Tensor([], pypto.DT_FP32),
    out:     pypto.Tensor([], pypto.DT_FP32),
):
    pypto.set_cube_tile_shapes([16, 16], [16, 16], [16, 16])
    res = pypto.matmul(col_mat, k_vec, pypto.DT_FP32)
    out.move(pypto.reshape(res, [res.shape[0]]))

X = torch.ones((2, 3), dtype=torch.float32, device=device)
X[:, 1] = 0
K = torch.ones((2, 3), dtype=torch.float32, device=device)

h, w = K.shape
outh = X.shape[0] - h + 1
outw = X.shape[1] - w + 1

# im2col: 提取所有 patch 并展平为行向量
cols = []
for i in range(outh):
    for j in range(outw):
        cols.append(X[i:i+h, j:j+w].reshape(-1))
col_mat = torch.stack(cols, 0)

# 卷积核展平为列向量
k_vec = K.reshape(-1, 1)

# 用 pypto.matmul 做矩阵乘法
out = torch.zeros(outh * outw, dtype=torch.float32, device=device)
practice_6_2_3_kernel(col_mat, k_vec, out)
print(out.reshape(outh, outw))

# 验证: 与原始 corr2d 结果一致
Y_ref = corr2d(X, K)
print(f'corr2d reference: {Y_ref}')

tensor([[4.]], device='npu:0')
corr2d reference: tensor([[4.]], device='npu:0')


---

## 练习 6.2.4

手工设计一些卷积核。
1. 二阶导数的核的形式是什么？
1. 积分的核的形式是什么？
1. 得到$d$次导数的最小核的大小是多少？

### 解答

**第1问：**

&emsp;&emsp;一维的二阶导数的核的形式是：

$$\begin{bmatrix}-1, & 2, & -1,\end{bmatrix}$$

&emsp;&emsp;这个一维卷积核可以用于计算信号或函数在每个点的二阶导数。它通过对信号进行两次微分来强调信号的曲率和变化率，通常用于边缘检测和特征检测等任务。

&emsp;&emsp;在二维情况下，可以使用以下卷积核来计算图像的二阶导数：

\begin{bmatrix}
    0, &  1, & 0 \\
    1, & -4, & 1 \\
    0, &  1, & 0 \\
\end{bmatrix}




**第2问：**

&emsp;&emsp;积分的核的形式是：

$$\begin{bmatrix}1 & 1 & 1 & \cdots & 1\end{bmatrix}$$


**第3问：**

&emsp;&emsp;得到 `𝑑` 次导数的最小核的大小是 $d+1$。例如，一阶导数的最小核大小为 $2$，二阶导数的最小核大小为 $3$，三阶导数的最小核大小为 $4$，以此类推。

---
## 参考答案来源
参考答案和 PyTorch 代码实现来源：[https://datawhalechina.github.io/d2l-ai-solutions-manual/#](https://datawhalechina.github.io/d2l-ai-solutions-manual/#)